<a href="https://colab.research.google.com/github/mafaiziyas/sp500-market-segmentation-fintech-analytics/blob/main/notebooks/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Task 1.1

In [37]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
import os
import sys
import json
import datetime
import urllib.request
from io import StringIO
import pandas as pd
import yfinance as yf

In [39]:
# helper func to print section headings
def heading(text):
    print("\n" + "=" * 60)
    print(text)
    print("=" * 60)

# settings
folder = "/content/drive/MyDrive/sp500 CW/data_snapshots"
os.makedirs(folder, exist_ok=True)
start_date = "2021-01-01"
end_date = "2025-01-01"
retrieval_date = datetime.date.today().isoformat()

In [40]:
# get S&P 500 tickers from Wikipedia

heading("2. Ticker List from Wikepedia")
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
html = urllib.request.urlopen(req).read().decode("utf-8")

tables = pd.read_html(StringIO(html))
wiki = tables[0]
sp500 = wiki[["Symbol", "Security", "GICS Sector", "GICS Sub-Industry"]].copy()

# Yahoo uses - instead of . (BRK.B becomes BRK-B)
sp500["YF_Ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
sp500.to_csv(f"{folder}/sp500_tickers_{retrieval_date}.csv", index=False)

print(f"Tickers listed : {len(sp500)}")
print(f"Unique tickers : {sp500['YF_Ticker'].nunique()}")
print(f"Saved to       : {folder}/sp500_tickers_{retrieval_date}.csv")

print("\nFirst 5 rows:")
print(sp500.head())


2. Ticker List from Wikepedia
Tickers listed : 503
Unique tickers : 503
Saved to       : /content/drive/MyDrive/sp500 CW/data_snapshots/sp500_tickers_2026-11-21.csv

First 5 rows:
  Symbol             Security             GICS Sector  \
0    MMM                   3M             Industrials   
1    AOS          A. O. Smith             Industrials   
2    ABT  Abbott Laboratories             Health Care   
3   ABBV               AbbVie             Health Care   
4    ACN            Accenture  Information Technology   

                GICS Sub-Industry YF_Ticker  
0        Industrial Conglomerates       MMM  
1               Building Products       AOS  
2           Health Care Equipment       ABT  
3                   Biotechnology      ABBV  
4  IT Consulting & Other Services       ACN  


In [41]:
# benchmark data
heading("3. BENCHMARK (^GSPC)")
bench = yf.download("^GSPC", start=start_date, end=end_date,
                    auto_adjust=False, progress=False)
if isinstance(bench.columns, pd.MultiIndex):
    bench.columns = bench.columns.get_level_values(0)
bench.to_csv(f"{folder}/benchmark_gspc.csv")

print(f"Trading days   : {len(bench)}")
print(f"First date     : {bench.index.min().date()}")
print(f"Last date      : {bench.index.max().date()}")

print("\nFirst 5 rows:")
print(bench.head())


3. BENCHMARK (^GSPC)
Trading days   : 1005
First date     : 2021-01-04
Last date      : 2024-12-31

First 5 rows:
Price         Adj Close        Close         High          Low         Open  \
Date                                                                          
2021-01-04  3700.649902  3700.649902  3769.989990  3662.709961  3764.610107   
2021-01-05  3726.860107  3726.860107  3737.830078  3695.070068  3698.020020   
2021-01-06  3748.139893  3748.139893  3783.040039  3705.340088  3712.199951   
2021-01-07  3803.790039  3803.790039  3811.550049  3764.709961  3764.709961   
2021-01-08  3824.679932  3824.679932  3826.689941  3783.600098  3815.050049   

Price           Volume  
Date                    
2021-01-04  5015000000  
2021-01-05  4591020000  
2021-01-06  6064110000  
2021-01-07  5099160000  
2021-01-08  4773040000  


In [42]:
# share data
heading("4. SHARE DATA DOWNLOAD")
tickers = sp500["YF_Ticker"].tolist()
shares = yf.download(tickers, start=start_date, end=end_date,
                     auto_adjust=False, actions=True,
                     group_by="ticker", progress=False)

first_day = shares.index.min()
last_day = shares.index.max()
print(f"First date     : {first_day.date()}")
print(f"Last date      : {last_day.date()}")
print(f"Trading days   : {len(shares)}")

# period check
assert first_day >= pd.Timestamp(start_date)
assert last_day < pd.Timestamp(end_date)
print("Period check   : passed")

# save one csv per field
fields = ["Open", "High", "Low", "Close", "Adj Close", "Volume",
          "Dividends", "Stock Splits"]
for f in fields:
    table = shares.xs(f, axis=1, level=1)
    table.to_csv(f"{folder}/shares_{f.replace(' ', '_').lower()}.csv")
print(f"Fields saved   : {', '.join(fields)}")

print(shares.shape)
print(shares["AAPL"].head())


4. SHARE DATA DOWNLOAD


ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['VYLR', 'HONA', 'FDXF', 'SNDK', 'Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


First date     : 2021-01-04
Last date      : 2024-12-31
Trading days   : 1005
Period check   : passed
Fields saved   : Open, High, Low, Close, Adj Close, Volume, Dividends, Stock Splits
(1005, 4014)
Price             Open        High         Low       Close   Adj Close  \
Date                                                                     
2021-01-04  133.520004  133.610001  126.760002  129.410004  125.537109   
2021-01-05  128.889999  131.740005  128.429993  131.009995  127.089211   
2021-01-06  127.720001  131.050003  126.379997  126.599998  122.811211   
2021-01-07  128.360001  131.630005  127.860001  130.919998  127.001930   
2021-01-08  132.429993  132.630005  130.229996  132.050003  128.098099   

Price          Volume  Dividends  Stock Splits  
Date                                            
2021-01-04  143301900        0.0           0.0  
2021-01-05   97664900        0.0           0.0  
2021-01-06  155088000        0.0           0.0  
2021-01-07  109578200        0.0     

In [43]:
# Validation check 1: tickers with no data

heading("5. TICKERS WITH NO PRICE DATA (EXCLUDED)")
no_data = []
for t in tickers:
    if t not in shares.columns.get_level_values(0):
        no_data.append([t, "not returned by Yahoo"])
    elif shares[t]["Close"].notna().sum() == 0:
        no_data.append([t, "no price data in period"])

excluded = pd.DataFrame(no_data, columns=["YF_Ticker", "reason"])

# add company name and the date it joined the index (evidence for the log)
excluded = excluded.merge(
    wiki[["Symbol", "Security", "Date added"]],
    left_on="YF_Ticker", right_on="Symbol", how="left"
).drop(columns="Symbol")
excluded["reason"] = "added to index after study period, no Yahoo price data"
excluded.to_csv(f"{folder}/acquisition_exclusions.csv", index=False)

print(f"Number excluded: {len(excluded)}\n")
print(excluded[["YF_Ticker", "Security", "Date added"]].to_string(index=False))



5. TICKERS WITH NO PRICE DATA (EXCLUDED)
Number excluded: 5

YF_Ticker            Security Date added
     FDXF       FedEx Freight 2026-06-01
     HONA Honeywell Aerospace 2026-06-29
        Q   Qnity Electronics 2025-11-03
     SNDK             Sandisk 2025-11-28
     VYLR               Vylor 2026-10-01


In [44]:
# Validation check 2: partial histories

heading("6. SHARES WITH PARTIAL HISTORY (KEPT FOR NOW)")
close = shares.xs("Close", axis=1, level=1)
counts = close.notna().sum()
short = counts[(counts > 0) & (counts < len(shares))].sort_values()

partial = short.reset_index()
partial.columns = ["YF_Ticker", "trading_days"]
partial["missing_days"] = len(shares) - partial["trading_days"]
partial.to_csv(f"{folder}/partial_history.csv", index=False)

print(f"Full history = {len(shares)} trading days")
print(f"Shares with partial history: {len(partial)}\n")
print(partial.to_string(index=False))



6. SHARES WITH PARTIAL HISTORY (KEPT FOR NOW)
Full history = 1005 trading days
Shares with partial history: 10

YF_Ticker  trading_days  missing_days
      GEV           193           812
     SOLV           194           811
     RDDT           197           808
     VLTO           313           692
     GEHC           513           492
      CEG           742           263
     HOOD           862           143
      APP           935            70
     COIN           936            69
      EXE           979            26


In [45]:
#The exclusion log

heading("5. TICKERS WITH NO PRICE DATA (EXCLUDED)")

no_data = []
for t in tickers:
    if t not in shares.columns.get_level_values(0):
        no_data.append([t, "not returned by Yahoo"])
    elif shares[t]["Close"].notna().sum() == 0:
        no_data.append([t, "no price data in period"])

excluded = pd.DataFrame(no_data, columns=["YF_Ticker", "reason"])
excluded = excluded.merge(wiki[["Symbol", "Security", "Date added"]],
                          left_on="YF_Ticker", right_on="Symbol", how="left").drop(columns="Symbol")
excluded["reason"] = "added to index after study period, no Yahoo price data"
excluded.to_csv(f"{folder}/acquisition_exclusions.csv", index=False)
print(excluded[["YF_Ticker", "Security", "Date added"]].to_string(index=False))


5. TICKERS WITH NO PRICE DATA (EXCLUDED)
YF_Ticker            Security Date added
     FDXF       FedEx Freight 2026-06-01
     HONA Honeywell Aerospace 2026-06-29
        Q   Qnity Electronics 2025-11-03
     SNDK             Sandisk 2025-11-28
     VYLR               Vylor 2026-10-01


#Task 1.2